# Étape 3 - Entraînement d'un modèle prédictif

On entraîne un premier modèle simple, on le **valide** avec une validation croisée
adaptée aux séries temporelles, puis on l'entraîne sur toutes les données.

## Modèle : forêt aléatoire

Une `RandomForestRegressor` = une moyenne de plusieurs arbres de décision.
Ici 10 arbres, avec une graine aléatoire fixée pour la reproductibilité.

## Validation croisée temporelle

On ne peut pas mélanger les dates au hasard (on prédirait le passé avec le futur).
`cross_validate` utilise `TimeSeriesSplit` : on entraîne sur un début de série,
on teste sur la suite, et on répète en agrandissant la fenêtre.

## 1. Importer les librairies

In [1]:
import sys
sys.path.append('..')
import yaml
import logging
import logging.config
import numpy as np
import pandas as pd
pd.set_option('display.min_rows', 500)
pd.set_option('display.max_rows', 500)
pd.set_option('display.max_columns', 500)
pd.set_option('display.width', 500)
pd.set_option('max_colwidth', 400)

from foodcast.domain.transform import etl
from foodcast.domain.feature_engineering import features_offline, features_online
from foodcast.domain.forecast import span_future, cross_validate, plotly_predictions
from foodcast.domain.multi_model import MultiModel
from sklearn.ensemble import RandomForestRegressor
import foodcast.settings as settings
import plotly.graph_objects as go

with open(settings.LOGGING_CONFIGURATION_FILE, 'r') as f:
    logging.config.dictConfig(yaml.safe_load(f.read()))

%load_ext autoreload
%autoreload 2

/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/mlflow/pyfunc/utils/data_validation.py:156: FutureWarning: Model's `predict` method contains invalid parameters: {'X'}. Only the following parameter names are allowed: context, model_input, and params. Note that invalid parameters will no longer be permitted in future versions.
  param_names = _check_func_signature(func, "predict")


************************************************************
USING default value : foodcast.settings.dev
************************************************************


## 2. Reprendre le jeu d'entraînement des étapes 1 et 2

In [9]:
# Reprise des étapes 1 et 2
df = etl(settings.DATA_DIR, 197, 200)
df = features_offline(df)
x_train = df.drop(columns=['cash_in']).set_index('order_date')
y_train = df[['order_date', 'cash_in']].set_index('order_date')['cash_in']
x_train.head()

2026-09-09 15:26:52 - /Users/moussa/Documents/ml_data_base/MlOps_1/forecasting/../foodcast/infrastructure/extract.py - INFO - extract: shape = (2158, 6)
2026-09-09 15:26:52 - /Users/moussa/Documents/ml_data_base/MlOps_1/forecasting/../foodcast/infrastructure/extract.py - INFO - extract: shape = (3247, 6)
2026-09-09 15:26:52 - /Users/moussa/Documents/ml_data_base/MlOps_1/forecasting/../foodcast/domain/transform.py - INFO - clean: shape = (380, 3)
2026-09-09 15:26:52 - /Users/moussa/Documents/ml_data_base/MlOps_1/forecasting/../foodcast/domain/transform.py - INFO - clean: shape = (565, 3)
2026-09-09 15:26:52 - /Users/moussa/Documents/ml_data_base/MlOps_1/forecasting/../foodcast/domain/transform.py - INFO - merge: shape = (945, 2)
2026-09-09 15:26:52 - /Users/moussa/Documents/ml_data_base/MlOps_1/forecasting/../foodcast/domain/transform.py - INFO - resample: shape = (659, 2)
2026-09-09 15:26:52 - /Users/moussa/Documents/ml_data_base/MlOps_1/forecasting/../foodcast/domain/transform.py - IN

,day_1,day_2,day_3,day_4,day_5,day_6,hour_cos_1,hour_sin_1,lag_1W
order_date,,,,,,,,,
2018-10-15 09:00:00,False,False,False,False,False,False,-0.707107,7.071068e-01,89.55
2018-10-15 10:00:00,False,False,False,False,False,False,-0.866025,5.000000e-01,0.00
2018-10-15 11:00:00,False,False,False,False,False,False,-0.965926,2.588190e-01,0.00
2018-10-15 12:00:00,False,False,False,False,False,False,-1.000000,1.224647e-16,0.00
2018-10-15 13:00:00,False,False,False,False,False,False,-0.965926,-2.588190e-01,0.00


## 3. Créer le modèle

10 arbres, `random_state` fixé (42 ici, n'importe quelle valeur fixe convient).

In [10]:
simple_model = RandomForestRegressor(n_estimators=10, random_state=42)
simple_model

,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",10
,"random_state random_state: int, RandomState instance or None, default=NoneControls both the randomness of the bootstrapping of the samples usedwhen building trees (if ``bootstrap=True``) and the sampling of thefeatures to consider when looking for the best split at each node(if ``max_features < n_features``).See :term:`Glossary <random_state>` for details.",42
,"criterion criterion: {""squared_error"", ""absolute_error"", ""poisson""}, default=""squared_error""The function to measure the quality of a split. Supported criteriaare ""squared_error"" for the mean squared error, which is equal tovariance reduction as feature selection criterion and minimizes the L2loss using the mean of each terminal node, ""absolute_error"" for the meanabsolute error, which minimizes the L1 loss using the median of each terminalnode, and ""poisson"" which uses reduction in Poisson deviance to find splits,also using the mean of each terminal node... versionadded:: 0.18 Mean Absolute Error (MAE) criterion... versionadded:: 1.0 Poisson criterion... versionchanged:: 1.9 Criterion `""friedman_mse""` was deprecated.",'squared_error'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=1.0The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None or 1.0, then `max_features=n_features`... note:: The default of 1.0 is equivalent to bagged trees and more randomness can be achieved by setting smaller values, e.g. 0.3... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to 1.0.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",1.0
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of

## 4. Regarder le code de `cross_validate`

In [4]:
cross_validate??

Signature:
cross_validate(
    model: sklearn.base.BaseEstimator,
    x: pandas.DataFrame,
    y: pandas.DataFrame,
    n_fold: int = 10,
) -> Tuple[<built-in function array>, pandas.DataFrame]
Source:   
def cross_validate(
    model: BaseEstimator,
    x: pd.DataFrame,
    y: pd.DataFrame,
    n_fold: int = 10
) -> Tuple[np.array, pd.DataFrame]:
    """
    Custom cross-validation, compatible with a sklearn TimeSeriesSplit.
    Return MAEs (Mean Absolute Errors) as well as a dataframe of predictions.

    Parameters
    ----------
    model : BaseEstimator
        Model to cross-validate.
    x : pd.DataFrame
        Input features of the training set.
    y : pd.DataFrame
        Input labels of the training set.
    n_fold : int
        Number of temporal cross-validation folds, by default 10.

    Returns
    -------
    Tuple[np.array, pd.DataFrame]
        maes: list of cross-validation MAEs (Mean Absolute Errors).
        preds: pd.DataFrame with one column 'y_true' and one or 

## 5. Valider le modèle (3 folds)

`cross_validate(model, x, y, n_fold)` renvoie :

- `maes` : un tableau des erreurs absolues moyennes (une par fold)
- `preds` : un dataframe des prédictions (colonne `y_pred_simple`)

In [5]:
maes, preds = cross_validate(simple_model, x_train, y_train, n_fold=3)
maes

2026-09-09 15:15:38 - foodcast.domain.forecast - INFO - Fold 0 - train shape: [(125, 9) - test shape: (122, 9)]
2026-09-09 15:15:38 - foodcast.domain.forecast - INFO - Fold 1 - train shape: [(247, 9) - test shape: (122, 9)]
2026-09-09 15:15:38 - foodcast.domain.forecast - INFO - Fold 2 - train shape: [(369, 9) - test shape: (122, 9)]


array([[23.94172131],
       [22.78698907],
       [32.40184836]])

**Question — unité de la MAE ?** Des dollars (même unité que `cash_in`).

**Est-ce un bon indicateur ici ?** Pas vraiment : le chiffre d'affaires est nul la nuit
et très élevé le soir. Une MAE globale mélange ces régimes très différents ;
une erreur *relative* ou calculée par tranche horaire serait plus parlante.

## 6. Regarder le code de `plotly_predictions`

In [6]:
plotly_predictions??

Signature:
plotly_predictions(
    preds: pandas.DataFrame,
    y: Optional[pandas.Series] = None,
) -> plotly.graph_objs._figure.Figure
Source:   
def plotly_predictions(preds: pd.DataFrame, y: Optional[pd.Series] = None) -> go.Figure:
    """
    (Plotly) Plot predictions and true labels if any.

    Parameters
    ----------
    preds : pd.DataFrame
        Predictions.
    y : Optional[pd.Series]
        True labels, by default None.

    Returns
    -------
    go.Figure
        The figure to plot.
    """
    fig = go.Figure()
    columns = [col for col in preds.columns if col.startswith('y_pred')]
    mini = preds[columns].min(axis=1)
    maxi = preds[columns].max(axis=1)
    if 'y_pred_simple' in preds.columns:
        fig.add_trace(
            go.Scatter(
                x=preds.index,
                y=preds['y_pred_simple'],
                line_color='red',
                name='simple predictions'
            )
        )
    if len(columns) > 1:
        fig.add_trace(
   

## 7. Tracer les prédictions de validation croisée face à la vérité

`plotly_predictions(preds, y)` : `preds` = prédictions, `y` = vraies valeurs.

In [7]:
plotly_predictions(preds, y_train)

2026-09-09 15:15:45 - foodcast.domain.forecast - INFO - plotly_predictions: target shape = (491,)
2026-09-09 15:15:45 - foodcast.domain.forecast - INFO - plotly_predictions: predictions shape = (366, 1)


## 8. Entraîner le modèle sur **tout** le jeu d'entraînement

Une fois la performance jugée acceptable, on ré-entraîne sur 100 % des données
avec la méthode `fit` de scikit-learn.

In [8]:
simple_model.fit(x_train, y_train)

,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",10
,"random_state random_state: int, RandomState instance or None, default=NoneControls both the randomness of the bootstrapping of the samples usedwhen building trees (if ``bootstrap=True``) and the sampling of thefeatures to consider when looking for the best split at each node(if ``max_features < n_features``).See :term:`Glossary <random_state>` for details.",42
,"criterion criterion: {""squared_error"", ""absolute_error"", ""poisson""}, default=""squared_error""The function to measure the quality of a split. Supported criteriaare ""squared_error"" for the mean squared error, which is equal tovariance reduction as feature selection criterion and minimizes the L2loss using the mean of each terminal node, ""absolute_error"" for the meanabsolute error, which minimizes the L1 loss using the median of each terminalnode, and ""poisson"" which uses reduction in Poisson deviance to find splits,also using the mean of each terminal node... versionadded:: 0.18 Mean Absolute Error (MAE) criterion... versionadded:: 1.0 Poisson criterion... versionchanged:: 1.9 Criterion `""friedman_mse""` was deprecated.",'squared_error'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=1.0The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None or 1.0, then `max_features=n_features`... note:: The default of 1.0 is equivalent to bagged trees and more randomness can be achieved by setting smaller values, e.g. 0.3... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to 1.0.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",1.0
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of

Le modèle est prêt à prédire. Mais pour prédire le futur, il faut d'abord
construire le jeu de prédiction.

➡️ Étape suivante : `04_feature_engineering_online.ipynb`